<a href="https://colab.research.google.com/github/laurena1083/lab-6.1/blob/main/1_4_health_care_survey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
import polars as pl

# Why use relative addresses?

In this notebook, we will illustrate

1. That relative addresses for loading data files works, but
2. Using absolute addresses for loading data files *will not*.

## Problem 1 - Load the `lat_long_example.csv` file using a relative address.

**Tasks.**
1. Open a terminal, start `nu`, and navigate to the root menu of your first/primary data repository,
2. Use `ls **/*` to get the relative address of `lat_long_example.csv`, and
3. Use `polars` to load and inspect these data using this relative path.

In [61]:
import pyarrow

In [62]:
relative_path = "./data/lat_long_examples.csv"

In [63]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [64]:
!mkdir -p sample_data

!curl https://raw.githubusercontent.com/laurena1083/lab-6.1/main/lat_long_examples.csv \
     -o ./sample_data/lat_long_examples.csv


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   206  100   206    0     0   1534      0 --:--:-- --:--:-- --:--:--  1537


In [65]:
(lat_long_examples :=
 spark.read.csv('./sample_data/lat_long_examples.csv', header=True)
)

DataFrame[City 1: string, Lat 1: string, Long 1: string, City 2: string, Lat 2: string, Long 2: string, Distance from Web (km): string]

## Problem 2 - Load the `lat_long_example.csv` file using a absolute address.

**Tasks.**
1. Open a terminal, start `nu`, and navigate to the root menu of *one of your first/primary data repository,
2. Use `glob **/*` to get the absolute address of `lat_long_example.csv`, and
3. Use `polars` to load and inspect these data.

In [66]:
absolute_path = "C: /Users/wi7536ul/OneDrive - Minnesota State/DSCI 326/labs/lab 6.1/lat_long_examples.csv"

## Illustrating the problem with absolute addresses

While the relative address in problem 1 points to the data IN THIS COPY of the repo, the absolute address points to the data in EXACTLY one of the copies of the repository. This becomes a problem if (A) anything changes in that repository, or (B) we are working on a different machine.

**Tasks.** To illustrate why this is a problem, do the following.

1. From your first/primary repository commit and push this notebook to GitHub,
2. Fetch and pull this notebook to another local copy of the repository,
3. In your file explorer (Files or Finder), move your first/primary repository into another folder, e.g., make a new folder and drag-and-drop the repo.
4. Rerun the cells in each local copy of the repository and document your findings in the WORD document.

In [67]:
!curl https://raw.githubusercontent.com/laurena1083/lab-6.1/main/health_survey_v2.csv \
     -o ./sample_data/health_survey_v2.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  195k  100  195k    0     0   852k      0 --:--:-- --:--:-- --:--:--  851k


In [68]:
(health_survey_v2 :=
 spark.read.csv('./sample_data/health_survey_v2.csv', header=True)
)

DataFrame[ID: string, F1: string, F5: string, F2: string, F1_1: string, F2_1: string, F6: string, F4: string, F3: string, F5_1: string, F1_2: string, F2_2: string, F6_1: string, F2_3: string, F4_1: string, F2_4: string, F5_2: string, F2_5: string, F6_2: string, F1_3: string, F2_6: string, F5_3: string, F4_2: string, F2_7: string, F3_1: string, F2_8: string, F5_4: string, F3_2: string, F1_4: string, F3_3: string, F1_5: string, F5_5: string, F6_3: string, F1_6: string, F5_6: string, F2_9: string, F3_4: string, F4_3: string, F2_10: string, F1_7: string, F6_4: string, F4_4: string, F5_7: string, F3_5: string, F2_11: string]

In [69]:
!curl https://raw.githubusercontent.com/laurena1083/lab-6.1/main/ReverseCodingItems_v2.csv \
     -o ./sample_data/ReverseCodingItems_v2.csv.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3157  100  3157    0     0  17435      0 --:--:-- --:--:-- --:--:-- 17441


In [70]:
(ReverseCodingItems_v2 :=
 spark.read.csv('./sample_data/ReverseCodingItems_v2.csv', header=True)
)

DataFrame[Question: string, Construct: string, Question # on Qualtrics Survey: string, Needs Reverse Coding?: string, Column Name: string]

In [71]:

from pyspark.sql.functions import col, explode, array, lit
survey_no_id = health_survey_v2.drop('ID')

melted = survey_no_id.selectExpr("stack({0}, {1}) as (variable, value)".format(
    len(survey_no_id.columns),
    ", ".join([f"'{c}', `{c}`" for c in survey_no_id.columns])
))
unique_values = melted.select("value").distinct()
value_list = [row["value"] for row in unique_values.collect()]

In [72]:
reg = {
    'Strongly Agree': 5,
    'Somewhat Agree': 4,
    'Strongly Disagree': 3,
    'Neither Agree nor Disagree': 2,
    'Somewhat Disagree': 1
}

In [73]:

rev = {'Strongly Agree':1,
         'Somewhat Agree':2,
         'Strongly Disagree':3,
         'Neither Agree nor Disagree':4,
         'Somewhat Disagree':5,
        }

In [74]:
q_col = list(filter(lambda c: c != 'ID', health_survey_v2.columns))
print(q_col)

['F1', 'F5', 'F2', 'F1_1', 'F2_1', 'F6', 'F4', 'F3', 'F5_1', 'F1_2', 'F2_2', 'F6_1', 'F2_3', 'F4_1', 'F2_4', 'F5_2', 'F2_5', 'F6_2', 'F1_3', 'F2_6', 'F5_3', 'F4_2', 'F2_7', 'F3_1', 'F2_8', 'F5_4', 'F3_2', 'F1_4', 'F3_3', 'F1_5', 'F5_5', 'F6_3', 'F1_6', 'F5_6', 'F2_9', 'F3_4', 'F4_3', 'F2_10', 'F1_7', 'F6_4', 'F4_4', 'F5_7', 'F3_5', 'F2_11']


In [75]:
health_survey_v2.columns


['ID',
 'F1',
 'F5',
 'F2',
 'F1_1',
 'F2_1',
 'F6',
 'F4',
 'F3',
 'F5_1',
 'F1_2',
 'F2_2',
 'F6_1',
 'F2_3',
 'F4_1',
 'F2_4',
 'F5_2',
 'F2_5',
 'F6_2',
 'F1_3',
 'F2_6',
 'F5_3',
 'F4_2',
 'F2_7',
 'F3_1',
 'F2_8',
 'F5_4',
 'F3_2',
 'F1_4',
 'F3_3',
 'F1_5',
 'F5_5',
 'F6_3',
 'F1_6',
 'F5_6',
 'F2_9',
 'F3_4',
 'F4_3',
 'F2_10',
 'F1_7',
 'F6_4',
 'F4_4',
 'F5_7',
 'F3_5',
 'F2_11']

In [92]:
melted = health_survey_v2.selectExpr(
    "ID",
    "stack({0}, {1}) as (Question, value)".format(
        len(q_col),
        ", ".join([f"'{col_name}', `{col_name}`" for col_name in q_col])
    )
)


In [93]:
from pyspark.sql.functions import col, when

melted = melted.withColumn("Response_str", col("value").cast("string"))

melted = melted.withColumn(
    "reg_coding",
    when(col("Response_str") == "Strongly Agree", 5)
    .when(col("Response_str") == "Somewhat Agree", 4)
    .when(col("Response_str") == "Strongly Disagree", 3)
    .when(col("Response_str") == "Neither Agree nor Disagree", 2)
    .when(col("Response_str") == "Somewhat Disagree", 1)
)

melted = melted.withColumn(
    "rev_coding",
    when(col("Response_str") == "Strongly Agree", 1)
    .when(col("Response_str") == "Somewhat Agree", 2)
    .when(col("Response_str") == "Strongly Disagree", 5)
    .when(col("Response_str") == "Neither Agree nor Disagree", 2)
    .when(col("Response_str") == "Somewhat Disagree", 4)
)


In [78]:
ReverseCodingItems_v2.columns

['Question',
 'Construct',
 'Question # on Qualtrics Survey',
 'Needs Reverse Coding?',
 'Column Name']

In [82]:
joined = melted.join(
    ReverseCodingItems_v2,
    melted["variable"] == ReverseCodingItems_v2["Column Name"],
    how="left"
)

In [94]:
from pyspark.sql.functions import when, split, col
reverse_clean = (
    ReverseCodingItems_v2.drop("Question")
    if "Question" in ReverseCodingItems_v2.columns
    else ReverseCodingItems_v2
)

joined = melted.join(
    reverse_clean,
    melted["Question"] == reverse_clean["Column Name"],
    how="left"
)


In [95]:
from pyspark.sql.functions import split

joined = joined.withColumn(
    "q_type",
    split(col("Question"), "_").getItem(0)
).withColumn(
    "coding",
    when(col("Needs Reverse Coding?") == "Yes", col("rev_coding"))
    .otherwise(col("reg_coding"))
)

In [96]:
import pandas as pd

health_survey_summary_df = (
    joined
    .groupBy("ID")
    .pivot("q_type")
    .sum("coding")
)

health_survey_summary_df.limit(5).toPandas()


,ID,F1,F2,F3,F4,F5,F6
0,125,32,46,22,16,26,18
1,124,37,55,21,16,36,19
2,7,35,56,20,18,24,21
3,51,33,52,24,17,24,23
4,169,35,49,17,18,22,14
